
# N10 Overnight – Active Learning × DAPT/SSL Interaction

## Ziel

Dieses Notebook prüft **gezielt** die Frage, ob Active Learning unter Distribution Shift
stärker mit dem zusätzlich domänenadaptierten Transformer (`DAPT_E2E`) wirkt als mit dem
ansonsten identischen, nicht zusätzlich domänenadaptierten Transformer (`T0_E2E`).

Der zentrale Test ist eine **Difference-in-Differences**:

\[
\Delta_{\mathrm{Interaktion}}
=
(DAPT_{\mathrm{Uncertainty}}-DAPT_{\mathrm{Random}})
-
(T0_{\mathrm{Uncertainty}}-T0_{\mathrm{Random}})
\]

### Kontrolliertes Design

- **10 Seeds:** 42, 52, 62, 72, 82, 242, 252, 262, 272, 282
- **Labelbudgets:** 10 %, 15 %, 20 %, 25 %
- **Strategien:** Random Sampling vs. Global Uncertainty Sampling
- **Modelle:** `T0_E2E` vs. `DAPT_E2E`
- **Primärer Betriebspunkt:** 0,5 % Ziel-FPR
- **Sekundär:** 1 % und 2 % Ziel-FPR
- **Stressszenarien:** Temporal, Domain OOD, Template OOD, Domain+Template OOD
- **Primärmetrik:** Recall am eingefrorenen Kalibrierungsschwellenwert
- **Sekundär:** empirische FPR, Precision, F1, AP, ROC-AUC

### Rechenersparnis

Die bereits vollständig berechneten **DAPT-Active-Learning-Trajektorien** aus dem produktiven
Loop werden **nicht neu trainiert**. Neu berechnet wird nur T0:

- 10-%-Startzustand einmal je Seed,
- Random 15/20/25 % auf **exakt denselben zufällig ausgewählten Labels wie DAPT**,
- T0-eigenes Uncertainty Sampling 15/20/25 %.

Damit sind maximal **70 T0-Fine-Tunings** erforderlich statt eines erneuten Gesamtlaufs.

### Wissenschaftliche Grenze

Eine positive Interaktion belegt, dass die **untersuchte Kombination aus DAPT und
Uncertainty-based Active Learning** unter dem gegebenen Protokoll stärker profitiert.
Sie beweist **nicht automatisch**, dass DAPT kausal „bessere Unsicherheiten“ erzeugt.
Dafür wären zusätzliche Unsicherheits-/Kalibrierungsanalysen erforderlich.


In [ ]:

# ============================================================
# 00 – Imports, Konfiguration, Output
# ============================================================
import os, gc, json, math, pickle, random, shutil, time, warnings, hashlib, zipfile
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
OUT_ROOT = Path("/kaggle/working/phreshphish_AL_SSL_INTERACTION_N10")
STATE_ROOT = OUT_ROOT / "t0_states"
TABLE_ROOT = OUT_ROOT / "tables"
FIG_ROOT = OUT_ROOT / "figures"
AUDIT_ROOT = OUT_ROOT / "audit"
for p in [OUT_ROOT, STATE_ROOT, TABLE_ROOT, FIG_ROOT, AUDIT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

ORIGINAL5 = [42, 52, 62, 72, 82]
REPLICATION5 = [242, 252, 262, 272, 282]
SEEDS = ORIGINAL5 + REPLICATION5
SEED_GROUP = {
    s: ("ORIGINAL5" if s in ORIGINAL5 else "REPLICATION5")
    for s in SEEDS
}

BUDGETS = [0.10, 0.15, 0.20, 0.25]
STEP_N = 200
STRATEGIES = ["RANDOM_GLOBAL", "UNCERTAINTY_GLOBAL"]

TARGET_FPRS = [0.005, 0.010, 0.020]
PRIMARY_FPR = 0.005
PRIMARY_BUDGET = 0.25

CALIBRATION_FRACTION = 0.30
CALIBRATION_SPLIT_SEED = 20260808
AP_BALANCE_SEED = 20260809
MAX_LENGTH = 256

# Exakt wie im produktiven/N10-E2E-Protokoll.
E2E_EPOCHS = 5
E2E_LR = 2e-5
E2E_WEIGHT_DECAY = 0.01
E2E_WARMUP_RATIO = 0.10
E2E_BATCH_CANDIDATES = [16, 8, 4]
E2E_SCORE_BATCH = 64

EXPECTED_SCENARIO_ROWS = {
    "TEMPORAL": 8000,
    "DOMAIN_OOD": 3858,
    "TEMPLATE_OOD": 6832,
    "DOMAIN_TEMPLATE_OOD": 3708,
}
FINAL_STRESS = [
    "TEMPORAL",
    "DOMAIN_OOD",
    "TEMPLATE_OOD",
    "DOMAIN_TEMPLATE_OOD",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False

print({
    "run": "AL_SSL_INTERACTION_N10_OVERNIGHT",
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seeds": SEEDS,
    "budgets": BUDGETS,
    "new_training": "T0 only",
    "dapt_al": "reuse old productive loop",
    "output": str(OUT_ROOT),
})


In [ ]:

# ============================================================
# 01 – Inputs robust finden (auch flache Kaggle-Datasets)
# ============================================================
def has_model_files(p):
    p = Path(p)
    return (
        p.is_dir()
        and (p / "config.json").exists()
        and (
            (p / "model.safetensors").exists()
            or (p / "pytorch_model.bin").exists()
        )
    )

def dataset_root(path):
    p = Path(path)
    parts = p.parts
    try:
        i = parts.index("datasets")
        if len(parts) > i + 2:
            return Path(*parts[:i+3])
    except ValueError:
        pass
    try:
        i = parts.index("input")
        if len(parts) > i + 1:
            return Path(*parts[:i+2])
    except ValueError:
        pass
    return p.parent

# Alter produktiver AL-Loop: DAPT-Ergebnisse + Selection-History.
loop_result_hits = list(INPUT_ROOT.rglob("LOOP_results_long.csv"))
loop_selection_hits = list(INPUT_ROOT.rglob("LOOP_selection_history.json"))

if not loop_result_hits or not loop_selection_hits:
    raise FileNotFoundError(
        "STOP: Alter produktiver Loop fehlt. Benötigt werden "
        "LOOP_results_long.csv und LOOP_selection_history.json "
        "(z.B. aus dem Dataset lpppll)."
    )

loop_roots = sorted(
    {dataset_root(p) for p in loop_result_hits + loop_selection_hits},
    key=lambda r: -(
        10 * len(list(r.rglob("LOOP_results_long.csv")))
        + 10 * len(list(r.rglob("LOOP_selection_history.json")))
        + len(list(r.rglob("DAPT_*_budget*.npz")))
    )
)
LOOP_SOURCE = loop_roots[0]

def find_in_loop(filename, required=True):
    hits = list(LOOP_SOURCE.rglob(filename))
    if not hits:
        if required:
            raise FileNotFoundError(
                f"STOP: {filename} fehlt im alten Loop-Dataset {LOOP_SOURCE}"
            )
        return None
    return sorted(hits, key=lambda p: (len(p.parts), len(str(p))))[0]

LOOP_RESULTS_PATH = find_in_loop("LOOP_results_long.csv")
LOOP_SELECTION_PATH = find_in_loop("LOOP_selection_history.json")
LOOP_ACQ_PATH = find_in_loop("LOOP_acquisition_audit.csv", required=False)

# FINAL-FREEZE Token-Caches.
def looks_like_freeze(p):
    p = Path(p)
    return (
        p.is_dir()
        and (p / "tokens" / "train_4k" / "input_ids.npy").exists()
        and (p / "tokens" / "calibration" / "input_ids.npy").exists()
        and (p / "tokens" / "iid_test" / "input_ids.npy").exists()
        and (p / "tokens" / "holdout_8k" / "input_ids.npy").exists()
    )

freeze_candidates = [p for p in INPUT_ROOT.rglob("*") if looks_like_freeze(p)]
if looks_like_freeze(INPUT_ROOT):
    freeze_candidates.append(INPUT_ROOT)
if not freeze_candidates:
    raise FileNotFoundError(
        "STOP: FINAL-FREEZE Token-Cache fehlt "
        "(tokens/train_4k, calibration, iid_test, holdout_8k)."
    )
FREEZE_ROOT = sorted(set(freeze_candidates), key=lambda p: len(str(p)))[0]

split_hits = list(INPUT_ROOT.rglob("split_roles_and_holdout_cache_v2_ram_safe.pkl"))
dapt_bundle_hits = list(INPUT_ROOT.rglob("dapt40k_bundle.pkl"))
if not split_hits or not dapt_bundle_hits:
    raise FileNotFoundError(
        "STOP: split_roles_and_holdout_cache_v2_ram_safe.pkl "
        "oder dapt40k_bundle.pkl fehlt."
    )
SPLIT_PATH = split_hits[0]
DAPT_BUNDLE_PATH = dapt_bundle_hits[0]

# Lokales RoBERTa-base für T0.
base_candidates = []
for p in INPUT_ROOT.rglob("*"):
    if p.is_dir() and has_model_files(p):
        low = str(p).lower()
        if "roberta-base" in low or p.name.lower() == "roberta-base":
            base_candidates.append(p)
if not base_candidates:
    raise FileNotFoundError(
        "STOP: lokales roberta-base für T0 wurde nicht gefunden."
    )
BASE_MODEL_DIR = sorted(base_candidates, key=lambda p: len(str(p)))[0]

# Optionaler Resume eines früheren Interaktionslaufs.
resume_hits = list(INPUT_ROOT.rglob("AL_SSL_INTERACTION_PROGRESS.json"))
RESUME_SOURCE = None
if resume_hits:
    r = dataset_root(resume_hits[0])
    # Nur kleine States/Audits übernehmen. Kein fremdes Training starten.
    for src_dir_name, dst in [
        ("t0_states", STATE_ROOT),
        ("audit", AUDIT_ROOT),
    ]:
        hits = [p for p in r.rglob(src_dir_name) if p.is_dir()]
        if hits:
            shutil.copytree(hits[0], dst, dirs_exist_ok=True)
    RESUME_SOURCE = str(r)

print(json.dumps({
    "loop_source": str(LOOP_SOURCE),
    "loop_results": str(LOOP_RESULTS_PATH),
    "loop_selection": str(LOOP_SELECTION_PATH),
    "freeze_root": str(FREEZE_ROOT),
    "split": str(SPLIT_PATH),
    "dapt_bundle": str(DAPT_BUNDLE_PATH),
    "base_model": str(BASE_MODEL_DIR),
    "resume_source": RESUME_SOURCE,
}, indent=2))


In [ ]:

# ============================================================
# 02 – Harte Validierung der vorhandenen DAPT-AL-Trajektorien
# ============================================================
OLD_LOOP = pd.read_csv(LOOP_RESULTS_PATH)
OLD_SELECTION = json.loads(
    LOOP_SELECTION_PATH.read_text(encoding="utf-8")
)

required_cols = {
    "strategy", "seed", "label_budget", "scenario", "target_fpr",
    "dapt_recall", "dapt_empirical_fpr", "dapt_precision",
    "dapt_f1", "dapt_ap", "dapt_auc"
}
missing_cols = sorted(required_cols - set(OLD_LOOP.columns))
if missing_cols:
    raise RuntimeError(
        f"STOP: LOOP_results_long.csv fehlen Spalten: {missing_cols}"
    )

# Direkte DAPT-Metriken sind wegen der Kaskaden-Reviewfraktionen mehrfach enthalten.
# Wir de-duplizieren später; zunächst prüfen wir Vollständigkeit.
for strategy in STRATEGIES:
    for seed in SEEDS:
        key = f"{strategy}__seed{seed}"
        if key not in OLD_SELECTION:
            raise RuntimeError(f"STOP: Selection-History fehlt: {key}")
        for b in BUDGETS:
            k = str(b)
            if k not in OLD_SELECTION[key]:
                raise RuntimeError(f"STOP: {key} Budget {b} fehlt.")
            idx = np.asarray(OLD_SELECTION[key][k], dtype=np.int32)
            expected_n = int(round(4000 * b))
            if len(idx) != expected_n:
                raise RuntimeError(
                    f"STOP: {key} Budget {b}: {len(idx)} statt {expected_n}"
                )

# Random-Labels müssen für DAPT und den neuen T0-Vergleich exakt wiederverwendbar sein.
# 10-%-Start muss zwischen Random und Uncertainty identisch sein.
for seed in SEEDS:
    a = np.sort(np.asarray(
        OLD_SELECTION[f"RANDOM_GLOBAL__seed{seed}"]["0.1"],
        dtype=np.int32
    ))
    b = np.sort(np.asarray(
        OLD_SELECTION[f"UNCERTAINTY_GLOBAL__seed{seed}"]["0.1"],
        dtype=np.int32
    ))
    if not np.array_equal(a, b):
        raise RuntimeError(
            f"STOP: 10%-Startzustand ist bei Seed {seed} nicht identisch."
        )

check = OLD_LOOP[
    OLD_LOOP.strategy.isin(STRATEGIES)
    & OLD_LOOP.seed.isin(SEEDS)
    & OLD_LOOP.label_budget.isin(BUDGETS)
    & OLD_LOOP.scenario.isin(
        ["IID_ORIGINAL", "IID_BALANCED"] + FINAL_STRESS
    )
    & OLD_LOOP.target_fpr.isin(TARGET_FPRS)
].copy()

if check.empty:
    raise RuntimeError("STOP: Keine passenden DAPT-AL-Ergebnisse gefunden.")

print({
    "preflight": "PASS",
    "dapt_al_reuse": True,
    "dapt_retraining": False,
    "t0_retraining_only": True,
    "selection_histories": len([
        k for k in OLD_SELECTION if any(k.startswith(s) for s in STRATEGIES)
    ]),
    "old_loop_rows_matching": len(check),
})


In [ ]:

# ============================================================
# 03 – Datenrollen, Szenarien und Token-Caches exakt rekonstruieren
# ============================================================
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def first_existing(obj, keys):
    if not isinstance(obj, dict):
        return None
    for k in keys:
        if k in obj:
            return obj[k]
    return None

split_payload = load_pickle(SPLIT_PATH)
dapt_payload = load_pickle(DAPT_BUNDLE_PATH)

train_df = first_existing(
    split_payload, ["train_df", "train", "supervised_train", "downstream_train"]
)
val_df = first_existing(
    split_payload, ["val_df", "validation_df", "validation", "val"]
)
holdout_df = first_existing(
    split_payload, ["final_holdout_clean", "final_holdout", "holdout_df", "holdout"]
)
pretrain_df = first_existing(
    dapt_payload, ["pretrain_large_df", "frame", "pretrain_df", "pretrain"]
)
dapt_holdout = first_existing(dapt_payload, ["final_holdout_clean"])

for name, obj in [
    ("train", train_df),
    ("validation", val_df),
    ("holdout", holdout_df),
    ("pretrain40k", pretrain_df),
]:
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f"{name}: kein DataFrame gefunden.")

train_df = train_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()
holdout_df = holdout_df.reset_index(drop=True).copy()
pretrain_df = pretrain_df.reset_index(drop=True).copy()

if len(train_df) != 4000:
    raise RuntimeError(f"STOP: Trainset {len(train_df)} statt 4000.")

if isinstance(dapt_holdout, pd.DataFrame):
    dapt_holdout = dapt_holdout.reset_index(drop=True)
    lookup = dapt_holdout.copy()
    lookup.index = lookup["sha256"].astype(str)
    hs = holdout_df["sha256"].astype(str)
    for c in [
        "near_duplicate_to_development",
        "min_simhash_distance_to_development",
        "template_seen_in_development",
    ]:
        if c in lookup.columns:
            mapped = hs.map(lookup[c])
            if mapped.isna().any():
                raise RuntimeError(
                    f"Holdout-Flag {c} konnte nicht vollständig gemappt werden."
                )
            holdout_df[c] = mapped.to_numpy()

cal_idx, iid_idx = train_test_split(
    np.arange(len(val_df)),
    test_size=1.0 - CALIBRATION_FRACTION,
    random_state=CALIBRATION_SPLIT_SEED,
    stratify=val_df["label"].to_numpy(),
)
calibration_df = val_df.iloc[np.sort(cal_idx)].reset_index(drop=True)
iid_df = val_df.iloc[np.sort(iid_idx)].reset_index(drop=True)

development_domains = set()
development_templates = set()
for frame in [pretrain_df, train_df, val_df]:
    development_domains.update(
        frame["domain"].fillna("").astype(str).tolist()
    )
    development_templates.update(
        frame["template_hash"].fillna("").astype(str).tolist()
    )
development_domains.discard("")
development_templates.discard("")

h_domain = holdout_df["domain"].fillna("").astype(str)
h_template = holdout_df["template_hash"].fillna("").astype(str)
domain_seen = h_domain.isin(development_domains).to_numpy()
template_seen = h_template.isin(development_templates).to_numpy()

if "near_duplicate_to_development" not in holdout_df.columns:
    raise RuntimeError("near_duplicate_to_development fehlt.")
near_dup = (
    holdout_df["near_duplicate_to_development"]
    .fillna(False)
    .astype(bool)
    .to_numpy()
)

def balanced_available_indices(frame, mask, seed):
    sub = frame.loc[np.asarray(mask)].copy()
    n_each = min(
        int((sub.label == 0).sum()),
        int((sub.label == 1).sum()),
    )
    p0 = sub[sub.label.eq(0)].sample(n=n_each, random_state=seed)
    p1 = sub[sub.label.eq(1)].sample(n=n_each, random_state=seed + 1)
    return np.sort(
        pd.concat([p0, p1]).index.to_numpy(dtype=np.int32)
    )

domain_new_mask = (~domain_seen) & h_domain.ne("").to_numpy()
template_ood_mask = (
    (~template_seen)
    & (~near_dup)
    & h_template.ne("").to_numpy()
)
domain_template_mask = domain_new_mask & template_ood_mask

stress_indices = {
    "TEMPORAL": np.arange(len(holdout_df), dtype=np.int32),
    "DOMAIN_OOD": balanced_available_indices(
        holdout_df, domain_new_mask, 108
    ),
    "TEMPLATE_OOD": balanced_available_indices(
        holdout_df, template_ood_mask, 109
    ),
    "DOMAIN_TEMPLATE_OOD": balanced_available_indices(
        holdout_df, domain_template_mask, 110
    ),
}
for name, expected in EXPECTED_SCENARIO_ROWS.items():
    if len(stress_indices[name]) != expected:
        raise RuntimeError(
            f"STOP: {name}: {len(stress_indices[name])} statt {expected}"
        )

iid_y = iid_df["label"].to_numpy(dtype=int)
iid_pos = np.flatnonzero(iid_y == 1)
iid_neg = np.flatnonzero(iid_y == 0)
n_each = min(len(iid_pos), len(iid_neg))
rng = np.random.default_rng(AP_BALANCE_SEED)
IID_BALANCED_IDX = np.sort(np.concatenate([
    rng.choice(iid_pos, n_each, replace=False),
    rng.choice(iid_neg, n_each, replace=False),
]).astype(np.int32))

SCENARIOS = {
    "IID_ORIGINAL": ("iid", np.arange(len(iid_df), dtype=np.int32)),
    "IID_BALANCED": ("iid", IID_BALANCED_IDX),
    "TEMPORAL": ("holdout", stress_indices["TEMPORAL"]),
    "DOMAIN_OOD": ("holdout", stress_indices["DOMAIN_OOD"]),
    "TEMPLATE_OOD": ("holdout", stress_indices["TEMPLATE_OOD"]),
    "DOMAIN_TEMPLATE_OOD": ("holdout", stress_indices["DOMAIN_TEMPLATE_OOD"]),
}

def load_token_split(folder):
    root = FREEZE_ROOT / "tokens" / folder
    return {
        "input_ids": np.load(root / "input_ids.npy", mmap_mode="r"),
        "attention_mask": np.load(root / "attention_mask.npy", mmap_mode="r"),
    }

TOKENS = {
    "train": load_token_split("train_4k"),
    "calibration": load_token_split("calibration"),
    "iid": load_token_split("iid_test"),
    "holdout": load_token_split("holdout_8k"),
}

expected_n = {
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid": len(iid_df),
    "holdout": len(holdout_df),
}
for split, arr in TOKENS.items():
    if arr["input_ids"].shape != (expected_n[split], MAX_LENGTH):
        raise RuntimeError(
            f"STOP: Token-Shape falsch: {split} {arr['input_ids'].shape}"
        )

y_train = train_df["label"].to_numpy(dtype=int)
ycal = calibration_df["label"].to_numpy(dtype=int)

print({
    "data_validation": "PASS",
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid": len(iid_df),
    "holdout": len(holdout_df),
    "stress": {k: len(v) for k, v in stress_indices.items()},
})


In [ ]:

# ============================================================
# 04 – Metriken, T0-Training, Scoring und Cache
# ============================================================
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def labeled_hash(indices):
    arr = np.sort(np.asarray(indices, dtype=np.int32))
    return hashlib.sha256(arr.tobytes()).hexdigest()

def threshold_for_target_fpr(y_true, score, target_fpr):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(score, dtype=float)
    neg = np.sort(s[y == 0])[::-1]
    allowed = int(math.floor(target_fpr * len(neg) + 1e-12))
    if allowed <= 0:
        return float(np.nextafter(neg[0], np.inf))
    if allowed >= len(neg):
        return float(-np.inf)
    return float(np.nextafter(neg[allowed], np.inf))

def metric_row(y, score, threshold):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()
    return {
        "average_precision": float(
            average_precision_score(y, score)
        ),
        "roc_auc": float(
            roc_auc_score(y, score)
        ) if len(np.unique(y)) == 2 else np.nan,
        "precision": float(
            precision_score(y, pred, zero_division=0)
        ),
        "recall": float(
            recall_score(y, pred, zero_division=0)
        ),
        "f1": float(
            f1_score(y, pred, zero_division=0)
        ),
        "empirical_fpr": float(
            fp / max(fp + tn, 1)
        ),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

def calibration_operating_points(ycal_, score):
    out = {}
    for target in TARGET_FPRS:
        thr = threshold_for_target_fpr(
            ycal_, score, target
        )
        m = metric_row(ycal_, score, thr)
        out[target] = {
            "threshold": thr,
            "calibration_empirical_fpr": m["empirical_fpr"],
            "calibration_recall": m["recall"],
        }
    return out

def scenario_score_view(iid_score, holdout_score, scenario):
    source, idx = SCENARIOS[scenario]
    if source == "iid":
        return (
            iid_df.iloc[idx]["label"].to_numpy(dtype=int),
            np.asarray(iid_score)[idx],
        )
    return (
        holdout_df.iloc[idx]["label"].to_numpy(dtype=int),
        np.asarray(holdout_score)[idx],
    )

class TokenSubsetDataset(Dataset):
    def __init__(self, arrays, labels, indices):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]
        self.labels = np.asarray(labels, dtype=np.int64)
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        return (
            torch.tensor(self.ids[j], dtype=torch.long),
            torch.tensor(self.mask[j], dtype=torch.long),
            torch.tensor(self.labels[j], dtype=torch.long),
        )

class TokenAllDataset(Dataset):
    def __init__(self, arrays):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return (
            torch.tensor(self.ids[i], dtype=torch.long),
            torch.tensor(self.mask[i], dtype=torch.long),
        )

@torch.no_grad()
def score_e2e_model(model, arrays, batch_size=E2E_SCORE_BATCH):
    ds = TokenAllDataset(arrays)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    model.eval()
    scores = []
    for ids, mask in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available(),
        ):
            logits = model(
                input_ids=ids,
                attention_mask=mask,
            ).logits
        scores.append(
            torch.softmax(
                logits.float(), dim=-1
            )[:, 1].cpu().numpy()
        )
    return np.concatenate(scores).astype(np.float32)

def train_t0_e2e(seed, round_i, labeled_idx):
    # Exakt dieselbe Seed-Logik wie im alten produktiven DAPT-Loop.
    train_seed = seed + 1000 * round_i
    last_error = None

    for batch_size in E2E_BATCH_CANDIDATES:
        try:
            set_all_seeds(train_seed)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = (
                AutoModelForSequenceClassification
                .from_pretrained(
                    str(BASE_MODEL_DIR),
                    num_labels=2,
                    local_files_only=True,
                    ignore_mismatched_sizes=True,
                )
                .to(DEVICE)
            )

            ds = TokenSubsetDataset(
                TOKENS["train"],
                y_train,
                labeled_idx,
            )
            g = torch.Generator()
            g.manual_seed(train_seed)
            loader = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=True,
                generator=g,
                num_workers=2,
                pin_memory=torch.cuda.is_available(),
            )

            grad_accum = max(1, 16 // batch_size)
            updates_per_epoch = math.ceil(
                len(loader) / grad_accum
            )
            total_updates = (
                updates_per_epoch * E2E_EPOCHS
            )

            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=E2E_LR,
                weight_decay=E2E_WEIGHT_DECAY,
            )
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(
                    round(
                        total_updates
                        * E2E_WARMUP_RATIO
                    )
                ),
                num_training_steps=total_updates,
            )
            scaler = torch.amp.GradScaler(
                "cuda",
                enabled=torch.cuda.is_available(),
            )

            started = time.perf_counter()
            model.train()
            optimizer.zero_grad(set_to_none=True)

            for epoch in range(E2E_EPOCHS):
                losses = []
                for step, (ids, mask, labels) in enumerate(loader):
                    ids = ids.to(DEVICE, non_blocking=True)
                    mask = mask.to(DEVICE, non_blocking=True)
                    labels = labels.to(DEVICE, non_blocking=True)

                    with torch.amp.autocast(
                        "cuda",
                        dtype=torch.float16,
                        enabled=torch.cuda.is_available(),
                    ):
                        loss = model(
                            input_ids=ids,
                            attention_mask=mask,
                            labels=labels,
                        ).loss / grad_accum

                    scaler.scale(loss).backward()
                    losses.append(
                        float(loss.detach().cpu())
                        * grad_accum
                    )

                    if (
                        (step + 1) % grad_accum == 0
                        or step + 1 == len(loader)
                    ):
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(), 1.0
                        )
                        scaler.step(optimizer)
                        scaler.update()
                        scheduler.step()
                        optimizer.zero_grad(set_to_none=True)

                print({
                    "model": "T0_E2E",
                    "seed": seed,
                    "round": round_i,
                    "epoch": epoch + 1,
                    "n_labeled": len(labeled_idx),
                    "loss": float(np.mean(losses)),
                })

            return (
                model,
                time.perf_counter() - started,
                batch_size,
                grad_accum,
            )

        except RuntimeError as e:
            last_error = e
            if "out of memory" not in str(e).lower():
                raise
            try:
                del model
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"T0 training failed seed={seed}, round={round_i}"
    ) from last_error

def state_filename(strategy, seed, budget):
    b = str(float(budget)).replace(".", "p")
    if budget == 0.10:
        strategy = "SHARED_INITIAL"
    return STATE_ROOT / f"T0_{strategy}_seed{seed}_budget{b}.npz"

def build_or_load_t0_state(strategy, seed, budget, round_i, labeled_idx):
    path = state_filename(strategy, seed, budget)
    expected_hash = labeled_hash(labeled_idx)

    if path.exists():
        d = np.load(path, allow_pickle=False)
        if str(d["labeled_hash"][0]) == expected_hash:
            print({
                "reuse": True,
                "strategy": strategy,
                "seed": seed,
                "budget": budget,
            })
            return {k: np.asarray(d[k]) for k in d.files}

    model, fit_seconds, bs, ga = train_t0_e2e(
        seed, round_i, labeled_idx
    )

    payload = {
        "labeled_hash": np.asarray([expected_hash]),
        "n_labeled": np.asarray(
            [len(labeled_idx)], dtype=np.int32
        ),
        "n_benign": np.asarray(
            [int((y_train[labeled_idx] == 0).sum())],
            dtype=np.int32,
        ),
        "n_phish": np.asarray(
            [int((y_train[labeled_idx] == 1).sum())],
            dtype=np.int32,
        ),
        "train_score": score_e2e_model(
            model, TOKENS["train"]
        ),
        "cal_score": score_e2e_model(
            model, TOKENS["calibration"]
        ),
        "iid_score": score_e2e_model(
            model, TOKENS["iid"]
        ),
        "holdout_score": score_e2e_model(
            model, TOKENS["holdout"]
        ),
        "fit_seconds": np.asarray(
            [fit_seconds], dtype=np.float64
        ),
        "batch_size": np.asarray(
            [bs], dtype=np.int32
        ),
        "grad_accum": np.asarray(
            [ga], dtype=np.int32
        ),
    }
    np.savez_compressed(path, **payload)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return payload

def write_progress(stage, **kwargs):
    payload = {
        "stage": stage,
        "utc": pd.Timestamp.utcnow().isoformat(),
        **kwargs,
    }
    (AUDIT_ROOT / "AL_SSL_INTERACTION_PROGRESS.json").write_text(
        json.dumps(payload, indent=2),
        encoding="utf-8",
    )


In [ ]:

# ============================================================
# 05 – T0 Active Learning: Random exakt gepaart + eigene Uncertainty
# ============================================================
T0_SELECTION_PATH = AUDIT_ROOT / "T0_selection_history.json"
T0_ACQ_PATH = AUDIT_ROOT / "T0_acquisition_audit.csv"

t0_selection = {}
t0_acq_rows = []

def old_indices(strategy, seed, budget):
    return np.sort(np.asarray(
        OLD_SELECTION[
            f"{strategy}__seed{seed}"
        ][str(budget)],
        dtype=np.int32,
    ))

def uncertainty_select(train_scores, unlabeled_idx, n_select):
    unlabeled_idx = np.asarray(
        unlabeled_idx, dtype=np.int32
    )
    uncertainty_distance = np.abs(
        np.asarray(train_scores, dtype=float)[unlabeled_idx]
        - 0.5
    )
    order = np.argsort(uncertainty_distance)
    chosen = unlabeled_idx[order[:n_select]]
    return np.sort(chosen.astype(np.int32)), uncertainty_distance[order[:n_select]]

for seed in SEEDS:
    initial = old_indices(
        "RANDOM_GLOBAL", seed, 0.10
    )

    # --------------------------------------------------------
    # A) RANDOM: exakt dieselben Labelindizes wie DAPT.
    # --------------------------------------------------------
    key_r = f"RANDOM_GLOBAL__seed{seed}"
    t0_selection[key_r] = {}

    for round_i, budget in enumerate(BUDGETS):
        labeled = old_indices(
            "RANDOM_GLOBAL", seed, budget
        )
        t0_selection[key_r][str(budget)] = labeled.tolist()

        state = build_or_load_t0_state(
            strategy="RANDOM_GLOBAL",
            seed=seed,
            budget=budget,
            round_i=round_i,
            labeled_idx=labeled,
        )

        if budget < BUDGETS[-1]:
            next_labeled = old_indices(
                "RANDOM_GLOBAL",
                seed,
                BUDGETS[round_i + 1],
            )
            chosen = np.setdiff1d(
                next_labeled,
                labeled,
                assume_unique=False,
            )
            revealed = y_train[chosen]
            t0_acq_rows.append({
                "model": "T0_E2E",
                "strategy": "RANDOM_GLOBAL",
                "seed": seed,
                "from_budget": budget,
                "to_budget": BUDGETS[round_i + 1],
                "selected_n": int(len(chosen)),
                "selected_benign": int(
                    (revealed == 0).sum()
                ),
                "selected_phish": int(
                    (revealed == 1).sum()
                ),
                "selected_phish_share": float(
                    (revealed == 1).mean()
                ),
                "mean_abs_distance_to_0p5": np.nan,
                "selection_source": (
                    "exact old DAPT random indices"
                ),
            })

        del state
        gc.collect()

    # --------------------------------------------------------
    # B) UNCERTAINTY: identischer 10%-Start, danach T0-eigen.
    # --------------------------------------------------------
    key_u = f"UNCERTAINTY_GLOBAL__seed{seed}"
    t0_selection[key_u] = {
        "0.1": initial.tolist()
    }
    labeled = initial.copy()

    for round_i, budget in enumerate(BUDGETS):
        expected_n = int(round(len(y_train) * budget))
        if len(labeled) != expected_n:
            raise RuntimeError(
                f"T0 AL Budget falsch seed={seed}, "
                f"budget={budget}: {len(labeled)} != {expected_n}"
            )

        state = build_or_load_t0_state(
            strategy="UNCERTAINTY_GLOBAL",
            seed=seed,
            budget=budget,
            round_i=round_i,
            labeled_idx=labeled,
        )

        t0_selection[key_u][str(budget)] = labeled.tolist()

        if budget < BUDGETS[-1]:
            unlabeled = np.setdiff1d(
                np.arange(len(y_train), dtype=np.int32),
                labeled,
            )
            chosen, distances = uncertainty_select(
                state["train_score"],
                unlabeled,
                STEP_N,
            )

            # Reviewer/Oracle-Labels werden erst nach Auswahl betrachtet.
            revealed = y_train[chosen]
            next_budget = BUDGETS[round_i + 1]

            t0_acq_rows.append({
                "model": "T0_E2E",
                "strategy": "UNCERTAINTY_GLOBAL",
                "seed": seed,
                "from_budget": budget,
                "to_budget": next_budget,
                "selected_n": int(len(chosen)),
                "selected_benign": int(
                    (revealed == 0).sum()
                ),
                "selected_phish": int(
                    (revealed == 1).sum()
                ),
                "selected_phish_share": float(
                    (revealed == 1).mean()
                ),
                "mean_abs_distance_to_0p5": float(
                    np.mean(distances)
                ),
                "selection_source": "T0 own uncertainty",
            })

            labeled = np.sort(
                np.concatenate([labeled, chosen])
            ).astype(np.int32)
            t0_selection[key_u][
                str(next_budget)
            ] = labeled.tolist()

        del state
        gc.collect()

    # Nach jedem Seed sichern -> Resume-fähig.
    T0_SELECTION_PATH.write_text(
        json.dumps(t0_selection),
        encoding="utf-8",
    )
    pd.DataFrame(t0_acq_rows).to_csv(
        T0_ACQ_PATH,
        index=False,
    )
    write_progress(
        "T0_SEED_COMPLETE",
        seed=seed,
        completed_seeds=[
            s for s in SEEDS
            if state_filename(
                "UNCERTAINTY_GLOBAL", s, 0.25
            ).exists()
        ],
    )

print({
    "t0_training_complete": True,
    "expected_states": 70,
    "actual_states": len(list(STATE_ROOT.glob("T0_*.npz"))),
    "acquisition_rows": len(t0_acq_rows),
})


In [ ]:

# ============================================================
# 06 – T0 auswerten + alte DAPT-Ergebnisse sauber übernehmen
# ============================================================
t0_rows = []

for strategy in STRATEGIES:
    for seed in SEEDS:
        for budget in BUDGETS:
            labeled = np.asarray(
                t0_selection[
                    f"{strategy}__seed{seed}"
                ][str(budget)],
                dtype=np.int32,
            )
            d = np.load(
                state_filename(strategy, seed, budget),
                allow_pickle=False,
            )
            if str(d["labeled_hash"][0]) != labeled_hash(labeled):
                raise RuntimeError(
                    f"State-Hash mismatch: {strategy}, {seed}, {budget}"
                )

            ops = calibration_operating_points(
                ycal,
                d["cal_score"],
            )

            for target in TARGET_FPRS:
                op = ops[target]
                for scenario in SCENARIOS:
                    y, score = scenario_score_view(
                        d["iid_score"],
                        d["holdout_score"],
                        scenario,
                    )
                    m = metric_row(
                        y, score, op["threshold"]
                    )

                    t0_rows.append({
                        "model": "T0_E2E",
                        "strategy": strategy,
                        "seed": seed,
                        "seed_group": SEED_GROUP[seed],
                        "label_budget": budget,
                        "n_labeled": int(d["n_labeled"][0]),
                        "labeled_benign": int(d["n_benign"][0]),
                        "labeled_phish": int(d["n_phish"][0]),
                        "scenario": scenario,
                        "target_fpr": target,
                        "threshold": op["threshold"],
                        "calibration_empirical_fpr": (
                            op["calibration_empirical_fpr"]
                        ),
                        "recall": m["recall"],
                        "empirical_fpr": m["empirical_fpr"],
                        "precision": m["precision"],
                        "f1": m["f1"],
                        "average_precision": m["average_precision"],
                        "roc_auc": m["roc_auc"],
                        "fit_seconds": float(d["fit_seconds"][0]),
                    })

T0_RESULTS = pd.DataFrame(t0_rows)
T0_RESULTS.to_csv(
    OUT_ROOT / "T0_ACTIVE_LEARNING_results_n10.csv",
    index=False,
)

# DAPT: wegen drei Reviewfraktionen in LOOP_results_long mehrfach vorhanden.
dapt = OLD_LOOP[
    OLD_LOOP.strategy.isin(STRATEGIES)
    & OLD_LOOP.seed.isin(SEEDS)
    & OLD_LOOP.label_budget.isin(BUDGETS)
    & OLD_LOOP.scenario.isin(SCENARIOS.keys())
    & OLD_LOOP.target_fpr.isin(TARGET_FPRS)
].copy()

# Alle direkten DAPT-Metriken müssen innerhalb der Review-Duplikate identisch sein.
direct_cols = [
    "dapt_recall",
    "dapt_empirical_fpr",
    "dapt_precision",
    "dapt_f1",
    "dapt_ap",
    "dapt_auc",
]
group_cols = [
    "strategy", "seed", "label_budget",
    "scenario", "target_fpr",
]
nunique = dapt.groupby(group_cols)[direct_cols].nunique(dropna=False)
if (nunique > 1).any().any():
    raise RuntimeError(
        "STOP: Direkte DAPT-Metriken unterscheiden sich unerwartet "
        "zwischen Reviewfraktionen."
    )

dapt = (
    dapt
    .sort_values(group_cols)
    .drop_duplicates(group_cols)
    .copy()
)

DAPT_RESULTS = pd.DataFrame({
    "model": "DAPT_E2E",
    "strategy": dapt["strategy"].to_numpy(),
    "seed": dapt["seed"].to_numpy(),
    "seed_group": dapt["seed"].map(SEED_GROUP).to_numpy(),
    "label_budget": dapt["label_budget"].to_numpy(),
    "n_labeled": dapt["n_labeled"].to_numpy(),
    "labeled_benign": dapt["labeled_benign"].to_numpy(),
    "labeled_phish": dapt["labeled_phish"].to_numpy(),
    "scenario": dapt["scenario"].to_numpy(),
    "target_fpr": dapt["target_fpr"].to_numpy(),
    "threshold": dapt["dapt_threshold"].to_numpy(),
    "calibration_empirical_fpr": dapt["dapt_calibration_fpr"].to_numpy(),
    "recall": dapt["dapt_recall"].to_numpy(),
    "empirical_fpr": dapt["dapt_empirical_fpr"].to_numpy(),
    "precision": dapt["dapt_precision"].to_numpy(),
    "f1": dapt["dapt_f1"].to_numpy(),
    "average_precision": dapt["dapt_ap"].to_numpy(),
    "roc_auc": dapt["dapt_auc"].to_numpy(),
    "fit_seconds": dapt["dapt_fit_seconds"].to_numpy(),
})

ALL_RESULTS = pd.concat(
    [T0_RESULTS, DAPT_RESULTS],
    ignore_index=True,
)
ALL_RESULTS.to_csv(
    OUT_ROOT / "AL_SSL_INTERACTION_results_long.csv",
    index=False,
)

expected = (
    2 * 2 * 10 * 4 * len(SCENARIOS) * 3
)
if len(ALL_RESULTS) != expected:
    raise RuntimeError(
        f"STOP: Ergebnismatrix {len(ALL_RESULTS)} != {expected}"
    )

print({
    "results_validation": "PASS",
    "rows": len(ALL_RESULTS),
    "t0_rows": len(T0_RESULTS),
    "dapt_rows_reused": len(DAPT_RESULTS),
})


In [ ]:

# ============================================================
# 07 – Statistik: AL-Gewinn je Modell + Difference-in-Differences
# ============================================================
def exact_signflip(diff):
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    obs = abs(diff.mean())
    vals = []
    for signs in product(
        [-1.0, 1.0],
        repeat=len(diff),
    ):
        vals.append(
            abs(
                np.mean(
                    diff * np.asarray(signs)
                )
            )
        )
    return float(
        np.mean(
            np.asarray(vals)
            >= obs - 1e-15
        )
    )

def paired_stats(diff):
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    n = len(diff)
    mean = float(diff.mean())
    sd = (
        float(diff.std(ddof=1))
        if n > 1 else np.nan
    )
    sem = (
        sd / math.sqrt(n)
        if n > 1 else np.nan
    )
    crit = (
        float(student_t.ppf(0.975, n - 1))
        if n > 1 else np.nan
    )
    return {
        "n_seeds": n,
        "mean_difference": mean,
        "sd_difference": sd,
        "ci95_low": (
            mean - crit * sem
            if n > 1 else np.nan
        ),
        "ci95_high": (
            mean + crit * sem
            if n > 1 else np.nan
        ),
        "exact_signflip_p_two_sided": exact_signflip(diff),
        "cohen_dz": (
            mean / sd
            if n > 1 and sd > 0 else np.nan
        ),
        "wins": int((diff > 1e-12).sum()),
        "ties": int(
            (np.abs(diff) <= 1e-12).sum()
        ),
        "losses": int((diff < -1e-12).sum()),
    }

def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        raw = pvals[idx]
        val = min(1.0, (m - rank) * raw)
        running = max(running, val)
        adjusted[idx] = running
    return adjusted

METRICS = [
    "recall",
    "empirical_fpr",
    "precision",
    "f1",
    "average_precision",
    "roc_auc",
]

stat_rows = []

for budget in BUDGETS:
    for target in TARGET_FPRS:
        for scenario in SCENARIOS:
            for metric in METRICS:
                gains = {}

                for model in ["T0_E2E", "DAPT_E2E"]:
                    u = ALL_RESULTS[
                        (ALL_RESULTS.model == model)
                        & (ALL_RESULTS.strategy == "UNCERTAINTY_GLOBAL")
                        & (ALL_RESULTS.label_budget == budget)
                        & (ALL_RESULTS.scenario == scenario)
                        & (ALL_RESULTS.target_fpr == target)
                    ].set_index("seed")

                    r = ALL_RESULTS[
                        (ALL_RESULTS.model == model)
                        & (ALL_RESULTS.strategy == "RANDOM_GLOBAL")
                        & (ALL_RESULTS.label_budget == budget)
                        & (ALL_RESULTS.scenario == scenario)
                        & (ALL_RESULTS.target_fpr == target)
                    ].set_index("seed")

                    gain = (
                        u.loc[SEEDS, metric].to_numpy()
                        - r.loc[SEEDS, metric].to_numpy()
                    )
                    gains[model] = gain

                    stat_rows.append({
                        "contrast": (
                            f"AL_gain_{model}_"
                            "UNCERTAINTY_minus_RANDOM"
                        ),
                        "model": model,
                        "scenario": scenario,
                        "label_budget": budget,
                        "target_fpr": target,
                        "metric": metric,
                        **paired_stats(gain),
                    })

                interaction = (
                    gains["DAPT_E2E"]
                    - gains["T0_E2E"]
                )
                stat_rows.append({
                    "contrast": (
                        "INTERACTION_"
                        "(DAPT_U-DAPT_R)-(T0_U-T0_R)"
                    ),
                    "model": "DAPT_vs_T0",
                    "scenario": scenario,
                    "label_budget": budget,
                    "target_fpr": target,
                    "metric": metric,
                    **paired_stats(interaction),
                })

STATS = pd.DataFrame(stat_rows)

# Holm-Korrektur nur für die vier vorab definierten primären
# Interaktionstests: Recall, 25 %, 0.5 %-FPR, vier Stressszenarien.
primary_mask = (
    STATS.contrast.str.startswith("INTERACTION_")
    & STATS.metric.eq("recall")
    & STATS.label_budget.eq(PRIMARY_BUDGET)
    & STATS.target_fpr.eq(PRIMARY_FPR)
    & STATS.scenario.isin(FINAL_STRESS)
)
STATS["holm_p_primary4"] = np.nan
primary_idx = STATS.index[primary_mask]
STATS.loc[
    primary_idx,
    "holm_p_primary4"
] = holm_adjust(
    STATS.loc[
        primary_idx,
        "exact_signflip_p_two_sided"
    ].to_numpy()
)

STATS.to_csv(
    OUT_ROOT / "AL_SSL_INTERACTION_statistics_n10.csv",
    index=False,
)

PRIMARY_STATS = STATS[primary_mask].copy()
PRIMARY_STATS.to_csv(
    TABLE_ROOT / "TABLE01_primary_interaction_25pct_fpr005.csv",
    index=False,
)

print(
    PRIMARY_STATS[
        [
            "scenario",
            "mean_difference",
            "ci95_low",
            "ci95_high",
            "exact_signflip_p_two_sided",
            "holm_p_primary4",
            "wins",
            "losses",
        ]
    ].to_string(index=False)
)


In [ ]:

# ============================================================
# 08 – Stress-Mittel je Seed + klare Bachelor-Kerntabelle
# ============================================================
primary = ALL_RESULTS[
    ALL_RESULTS.scenario.isin(FINAL_STRESS)
    & ALL_RESULTS.target_fpr.eq(PRIMARY_FPR)
].copy()

stress_seed = (
    primary
    .groupby(
        [
            "model",
            "strategy",
            "seed",
            "label_budget",
        ],
        as_index=False,
    )
    .agg(
        recall=("recall", "mean"),
        empirical_fpr=("empirical_fpr", "mean"),
        average_precision=("average_precision", "mean"),
        roc_auc=("roc_auc", "mean"),
    )
)

stress_summary = (
    stress_seed
    .groupby(
        ["model", "strategy", "label_budget"],
        as_index=False,
    )
    .agg(
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        fpr_mean=("empirical_fpr", "mean"),
        ap_mean=("average_precision", "mean"),
        auc_mean=("roc_auc", "mean"),
    )
)
stress_summary.to_csv(
    TABLE_ROOT / "TABLE02_stress_mean_by_budget.csv",
    index=False,
)

stress_interaction_rows = []
for budget in BUDGETS:
    gains = {}
    for model in ["T0_E2E", "DAPT_E2E"]:
        u = stress_seed[
            (stress_seed.model == model)
            & (stress_seed.strategy == "UNCERTAINTY_GLOBAL")
            & (stress_seed.label_budget == budget)
        ].set_index("seed")
        r = stress_seed[
            (stress_seed.model == model)
            & (stress_seed.strategy == "RANDOM_GLOBAL")
            & (stress_seed.label_budget == budget)
        ].set_index("seed")
        gains[model] = (
            u.loc[SEEDS, "recall"].to_numpy()
            - r.loc[SEEDS, "recall"].to_numpy()
        )

        stress_interaction_rows.append({
            "contrast": f"AL_gain_{model}",
            "label_budget": budget,
            **paired_stats(gains[model]),
        })

    interaction = (
        gains["DAPT_E2E"] - gains["T0_E2E"]
    )
    stress_interaction_rows.append({
        "contrast": "INTERACTION_DAPT_AL_gain_minus_T0_AL_gain",
        "label_budget": budget,
        **paired_stats(interaction),
    })

STRESS_INTERACTION = pd.DataFrame(
    stress_interaction_rows
)
STRESS_INTERACTION.to_csv(
    TABLE_ROOT / "TABLE03_stress_mean_interaction.csv",
    index=False,
)

# Lesbare 25%-Kerntabelle
core25 = stress_summary[
    stress_summary.label_budget.eq(0.25)
].copy()
core25["strategy_short"] = core25["strategy"].map({
    "RANDOM_GLOBAL": "Random",
    "UNCERTAINTY_GLOBAL": "Uncertainty",
})
core25 = core25[
    [
        "model",
        "strategy_short",
        "recall_mean",
        "fpr_mean",
        "ap_mean",
        "auc_mean",
    ]
].sort_values(["model", "strategy_short"])
core25.to_csv(
    TABLE_ROOT / "TABLE04_core_25pct_stress_mean.csv",
    index=False,
)

print("\n25%-Stressmittel:")
print(core25.to_string(index=False))

print("\nStressmittel-Interaktion:")
print(
    STRESS_INTERACTION[
        STRESS_INTERACTION.contrast.eq(
            "INTERACTION_DAPT_AL_gain_minus_T0_AL_gain"
        )
    ].to_string(index=False)
)


In [ ]:

# ============================================================
# 09 – Zusätzliche Kontrollauswertung:
#      DAPT-vs-T0-Abstand unter Random vs Uncertainty
# ============================================================
model_gap_rows = []

for strategy in STRATEGIES:
    for budget in BUDGETS:
        for target in TARGET_FPRS:
            for scenario in SCENARIOS:
                for metric in METRICS:
                    d = ALL_RESULTS[
                        (ALL_RESULTS.model == "DAPT_E2E")
                        & (ALL_RESULTS.strategy == strategy)
                        & (ALL_RESULTS.label_budget == budget)
                        & (ALL_RESULTS.scenario == scenario)
                        & (ALL_RESULTS.target_fpr == target)
                    ].set_index("seed")

                    t = ALL_RESULTS[
                        (ALL_RESULTS.model == "T0_E2E")
                        & (ALL_RESULTS.strategy == strategy)
                        & (ALL_RESULTS.label_budget == budget)
                        & (ALL_RESULTS.scenario == scenario)
                        & (ALL_RESULTS.target_fpr == target)
                    ].set_index("seed")

                    diff = (
                        d.loc[SEEDS, metric].to_numpy()
                        - t.loc[SEEDS, metric].to_numpy()
                    )
                    model_gap_rows.append({
                        "contrast": "DAPT_minus_T0",
                        "strategy": strategy,
                        "scenario": scenario,
                        "label_budget": budget,
                        "target_fpr": target,
                        "metric": metric,
                        **paired_stats(diff),
                    })

MODEL_GAPS = pd.DataFrame(model_gap_rows)
MODEL_GAPS.to_csv(
    OUT_ROOT / "DAPT_T0_gaps_by_AL_strategy.csv",
    index=False,
)

gap_primary = MODEL_GAPS[
    MODEL_GAPS.scenario.isin(FINAL_STRESS)
    & MODEL_GAPS.label_budget.eq(0.25)
    & MODEL_GAPS.target_fpr.eq(PRIMARY_FPR)
    & MODEL_GAPS.metric.eq("recall")
].copy()
gap_primary.to_csv(
    TABLE_ROOT / "TABLE05_DAPT_T0_gap_random_vs_uncertainty.csv",
    index=False,
)

print(gap_primary[
    [
        "strategy",
        "scenario",
        "mean_difference",
        "ci95_low",
        "ci95_high",
        "exact_signflip_p_two_sided",
        "wins",
        "losses",
    ]
].to_string(index=False))


In [ ]:

# ============================================================
# 10 – Abbildungen ohne Modellinterpretation vorwegzunehmen
# ============================================================
# FIG 1: Stressmittel-Recall über Labelbudgets
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for (model, strategy), g in stress_summary.groupby(
    ["model", "strategy"]
):
    g = g.sort_values("label_budget")
    ax.plot(
        g["label_budget"] * 100,
        g["recall_mean"] * 100,
        marker="o",
        label=f"{model} | {strategy.replace('_GLOBAL','')}",
    )
ax.set_xlabel("Gelabelter Trainingsanteil [%]")
ax.set_ylabel("Recall über vier Stressszenarien [%]")
ax.set_title(
    "Random vs. Uncertainty: T0 und DAPT bei 0,5 % Ziel-FPR"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_ROOT / "FIG01_AL_SSL_stress_recall_budget.png",
    dpi=220,
)
fig.savefig(
    FIG_ROOT / "FIG01_AL_SSL_stress_recall_budget.pdf"
)
plt.close(fig)

# FIG 2: AL-Gewinn bei 25 % pro Stressszenario
plot_rows = []
for model in ["T0_E2E", "DAPT_E2E"]:
    for scenario in FINAL_STRESS:
        u = ALL_RESULTS[
            (ALL_RESULTS.model == model)
            & (ALL_RESULTS.strategy == "UNCERTAINTY_GLOBAL")
            & (ALL_RESULTS.label_budget == 0.25)
            & (ALL_RESULTS.scenario == scenario)
            & (ALL_RESULTS.target_fpr == PRIMARY_FPR)
        ].set_index("seed")
        r = ALL_RESULTS[
            (ALL_RESULTS.model == model)
            & (ALL_RESULTS.strategy == "RANDOM_GLOBAL")
            & (ALL_RESULTS.label_budget == 0.25)
            & (ALL_RESULTS.scenario == scenario)
            & (ALL_RESULTS.target_fpr == PRIMARY_FPR)
        ].set_index("seed")
        diff = (
            u.loc[SEEDS, "recall"].to_numpy()
            - r.loc[SEEDS, "recall"].to_numpy()
        )
        st = paired_stats(diff)
        plot_rows.append({
            "model": model,
            "scenario": scenario,
            "gain_pp": 100 * st["mean_difference"],
            "ci_low_pp": 100 * st["ci95_low"],
            "ci_high_pp": 100 * st["ci95_high"],
        })

PLOT_GAIN = pd.DataFrame(plot_rows)
PLOT_GAIN.to_csv(
    TABLE_ROOT / "TABLE06_AL_gain_plot_data.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(9.2, 5.3))
x = np.arange(len(FINAL_STRESS))
width = 0.36
for j, model in enumerate(["T0_E2E", "DAPT_E2E"]):
    p = PLOT_GAIN[
        PLOT_GAIN.model.eq(model)
    ].set_index("scenario").loc[FINAL_STRESS]
    pos = x + (j - 0.5) * width
    y = p["gain_pp"].to_numpy()
    lo = y - p["ci_low_pp"].to_numpy()
    hi = p["ci_high_pp"].to_numpy() - y
    ax.bar(pos, y, width=width, label=model)
    ax.errorbar(
        pos, y, yerr=np.vstack([lo, hi]),
        fmt="none", capsize=3
    )
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(FINAL_STRESS, rotation=20, ha="right")
ax.set_ylabel("Recall-Gewinn Uncertainty vs. Random [pp]")
ax.set_title(
    "Active-Learning-Gewinn bei 25 % Labels und 0,5 % Ziel-FPR"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_ROOT / "FIG02_AL_gain_T0_vs_DAPT.png",
    dpi=220,
)
fig.savefig(
    FIG_ROOT / "FIG02_AL_gain_T0_vs_DAPT.pdf"
)
plt.close(fig)

# FIG 3: eigentliche Interaktion
p = PRIMARY_STATS.set_index("scenario").loc[FINAL_STRESS]
y = 100 * p["mean_difference"].to_numpy()
lo = y - 100 * p["ci95_low"].to_numpy()
hi = 100 * p["ci95_high"].to_numpy() - y

fig, ax = plt.subplots(figsize=(8.8, 5.0))
x = np.arange(len(FINAL_STRESS))
ax.errorbar(
    x, y, yerr=np.vstack([lo, hi]),
    fmt="o", capsize=4
)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(FINAL_STRESS, rotation=20, ha="right")
ax.set_ylabel(
    "Interaktion: zusätzlicher DAPT-AL-Gewinn gegenüber T0 [pp]"
)
ax.set_title(
    "DAPT × Active Learning: Difference-in-Differences"
)
fig.tight_layout()
fig.savefig(
    FIG_ROOT / "FIG03_AL_SSL_interaction.png",
    dpi=220,
)
fig.savefig(
    FIG_ROOT / "FIG03_AL_SSL_interaction.pdf"
)
plt.close(fig)

print({
    "figures_png": len(list(FIG_ROOT.glob("*.png"))),
    "figures_pdf": len(list(FIG_ROOT.glob("*.pdf"))),
})


In [ ]:

# ============================================================
# 11 – Automatische Ergebnisentscheidung + Completion + ZIP
# ============================================================
primary_sorted = (
    PRIMARY_STATS
    .set_index("scenario")
    .loc[FINAL_STRESS]
    .reset_index()
)

positive_all4 = bool(
    (primary_sorted["mean_difference"] > 0).all()
)
ci_positive_all4 = bool(
    (primary_sorted["ci95_low"] > 0).all()
)
holm_sig_all4 = bool(
    (primary_sorted["holm_p_primary4"] < 0.05).all()
)

stress25 = STRESS_INTERACTION[
    STRESS_INTERACTION.contrast.eq(
        "INTERACTION_DAPT_AL_gain_minus_T0_AL_gain"
    )
    & STRESS_INTERACTION.label_budget.eq(0.25)
].iloc[0]

if (
    stress25["mean_difference"] > 0
    and stress25["ci95_low"] > 0
    and stress25["exact_signflip_p_two_sided"] < 0.05
):
    automatic_reading = (
        "SUPPORT: Im Mittel über die vier Stressszenarien ist der "
        "Recall-Gewinn durch Uncertainty Sampling bei DAPT größer als "
        "bei T0. Dies stützt eine positive DAPT×AL-Interaktion im "
        "untersuchten Protokoll."
    )
elif stress25["mean_difference"] > 0:
    automatic_reading = (
        "DESCRIPTIVE_ONLY: Die Interaktion ist im Stressmittel positiv, "
        "aber statistisch nicht hinreichend abgesichert. Active Learning "
        "darf nicht als nachgewiesener SSL-Verstärker formuliert werden."
    )
else:
    automatic_reading = (
        "NO_SUPPORT: Der Active-Learning-Gewinn ist im Stressmittel bei "
        "DAPT nicht größer als bei T0. Active Learning bleibt ein "
        "allgemeiner Lifecycle-Hebel und kein SSL-Verstärker."
    )

completion = {
    "status": "COMPLETE",
    "run": "AL_SSL_INTERACTION_N10_OVERNIGHT",
    "seeds": SEEDS,
    "models": ["T0_E2E", "DAPT_E2E"],
    "strategies": STRATEGIES,
    "budgets": BUDGETS,
    "target_fprs": TARGET_FPRS,
    "primary": {
        "metric": "recall",
        "label_budget": PRIMARY_BUDGET,
        "target_fpr": PRIMARY_FPR,
        "scenarios": FINAL_STRESS,
        "interaction_definition": (
            "(DAPT_UNCERTAINTY-DAPT_RANDOM)"
            "-(T0_UNCERTAINTY-T0_RANDOM)"
        ),
    },
    "reuse": {
        "dapt_active_learning_retrained": False,
        "dapt_source": str(LOOP_SOURCE),
        "t0_newly_trained": True,
        "random_label_indices_identical_between_T0_and_DAPT": True,
        "initial_10pct_identical_across_strategies_and_models": True,
    },
    "checks": {
        "expected_t0_state_files": 70,
        "actual_t0_state_files": len(list(STATE_ROOT.glob("T0_*.npz"))),
        "result_rows": len(ALL_RESULTS),
        "statistics_rows": len(STATS),
        "primary_positive_all4": positive_all4,
        "primary_ci_positive_all4": ci_positive_all4,
        "primary_holm_significant_all4": holm_sig_all4,
    },
    "stress_mean_25pct_interaction": {
        k: (
            float(stress25[k])
            if isinstance(stress25[k], (int, float, np.integer, np.floating))
            and pd.notna(stress25[k])
            else stress25[k]
        )
        for k in [
            "mean_difference",
            "ci95_low",
            "ci95_high",
            "exact_signflip_p_two_sided",
            "wins",
            "ties",
            "losses",
        ]
    },
    "automatic_reading": automatic_reading,
    "scientific_boundary": (
        "Positive interaction supports the investigated DAPT + "
        "uncertainty-sampling combination; it does not by itself prove "
        "that DAPT causally improves uncertainty calibration."
    ),
}

(OUT_ROOT / "AL_SSL_INTERACTION_COMPLETE.json").write_text(
    json.dumps(completion, indent=2, default=str),
    encoding="utf-8",
)

readme = f"""# AL × SSL Interaction N10

## Primärer Test
Difference-in-Differences:
(DAPT_Uncertainty - DAPT_Random) - (T0_Uncertainty - T0_Random)

## Automatische Einordnung
{automatic_reading}

## Methodische Grenze
Der Interaktionstest vergleicht die operationalen Lernstrategien.
Da T0 und DAPT unter Uncertainty Sampling modellabhängig unterschiedliche
Fälle auswählen können, darf eine positive Interaktion nicht ohne zusätzliche
Analyse als kausaler Nachweis besserer DAPT-Unsicherheitskalibrierung
interpretiert werden.

## Reuse
- DAPT-AL-Ergebnisse: vorhandener produktiver Loop
- T0: neu gerechnet
- Random-Labelsets: T0 und DAPT exakt identisch
- 10%-Start: über alle Bedingungen identisch
"""
(OUT_ROOT / "README_INTERACTION.md").write_text(
    readme, encoding="utf-8"
)

# Kleines Ergebnispaket – keine großen State-NPZs.
zip_path = Path(
    "/kaggle/working/"
    "phreshphish_AL_SSL_INTERACTION_N10_ANALYSIS.zip"
)
with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for p in OUT_ROOT.rglob("*"):
        if not p.is_file():
            continue
        if STATE_ROOT in p.parents:
            continue
        z.write(
            p,
            arcname=str(p.relative_to(OUT_ROOT)),
        )

write_progress(
    "COMPLETE",
    completion_file=str(
        OUT_ROOT / "AL_SSL_INTERACTION_COMPLETE.json"
    ),
    analysis_zip=str(zip_path),
)

print(json.dumps(completion, indent=2, default=str))
print({"ANALYSIS_ZIP": str(zip_path)})



## Interpretation für die Bachelorarbeit

Die **primäre SSL-Forschungsfrage bleibt unverändert** und wird weiterhin durch den
kontrollierten T0-vs.-DAPT-Vergleich des N10-Kerns beantwortet.

Dieses Experiment ist eine gezielte Vertiefung zu FF3:

- **Positive, seed-stabile Interaktion:** Hinweise darauf, dass der Nutzen gezielter
  Labelakquisition unter dem untersuchten Protokoll bei DAPT stärker ausfällt als bei T0.
- **Ähnliche AL-Gewinne:** Active Learning wirkt primär als allgemeiner Labelakquisitionshebel.
- **Stärkerer T0-Gewinn:** Keine Evidenz für einen SSL-spezifischen Verstärkungseffekt.

Unabhängig vom Ergebnis darf aus diesem Experiment nicht ohne zusätzliche
Unsicherheits-/Kalibrierungsanalyse abgeleitet werden, *warum* eine Interaktion entsteht.
